# LeetCode 2099 — Find Subsequence of Length K With the Largest Sum bằng Python

**Link đề:** https://leetcode.com/problems/find-subsequence-of-length-k-with-the-largest-sum/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table, Sorting, Heap (Priority Queue)

```python
class Solution:
    def maxSubsequence(self, nums: list[int], k: int) -> list[int]:
        ...
```

## Mục tiêu học
1. Phân biệt **subsequence** (dãy con giữ thứ tự, không cần liên tiếp) với **subarray** (đoạn liên tiếp).
2. Hiểu trực giác **tham lam**: tổng lớn nhất ⇒ lấy `k` phần tử lớn nhất; khó khăn là **giữ đúng thứ tự gốc**.
3. Kỹ thuật **sắp xếp kèm chỉ số** (`sorted(range(n), key=...)`, `enumerate`), `heapq` (`nlargest`, min-heap kích thước k), xử lý **giá trị trùng** bằng đếm ngưỡng.
4. Đi từ baseline "xóa dần phần tử nhỏ nhất" `O(n²)` → sắp xếp `O(n log n)` → heap `O(n log k)`.
5. Viết test cho bài **có nhiều đáp án đúng** (kiểm tra tính chất, không so khớp cứng).

## 1. Đề bài và cách hiểu đúng

Cho mảng `nums` và số `k`. Tìm một **dãy con (subsequence)** độ dài `k` có **tổng lớn nhất**. Trả về **bất kỳ** dãy nào như vậy.

*Subsequence* = xóa đi một số phần tử (hoặc không xóa), **giữ nguyên thứ tự** các phần tử còn lại.

| `nums` | `k` | Một đáp án | Tổng |
|---|:---:|---|:---:|
| `[2,1,3,3]` | 2 | `[3,3]` | 6 |
| `[-1,-2,3,4]` | 3 | `[-1,3,4]` | 6 |
| `[3,4,3,3]` | 2 | `[3,4]` (hoặc `[4,3]`) | 7 |

**Constraints:** `1 <= len(nums) <= 1000`, `-10^5 <= nums[i] <= 10^5`, `1 <= k <= len(nums)`.

Ví dụ 2 quan trọng: đáp án là `[-1, 3, 4]` chứ **không** phải `[3, 4, -1]` — phải theo thứ tự xuất hiện trong mảng gốc.

In [1]:
examples = [
    ([2, 1, 3, 3], 2, [3, 3]),
    ([-1, -2, 3, 4], 3, [-1, 3, 4]),
    ([3, 4, 3, 3], 2, [3, 4]),
]
for nums, k, ans in examples:
    print(nums, "k =", k, "-> một đáp án:", ans, "tổng", sum(ans))

[2, 1, 3, 3] k = 2 -> một đáp án: [3, 3] tổng 6
[-1, -2, 3, 4] k = 3 -> một đáp án: [-1, 3, 4] tổng 6
[3, 4, 3, 3] k = 2 -> một đáp án: [3, 4] tổng 7


### Subsequence khác subarray thế nào?

Với `[5, 1, 7]`:
- `[5, 7]` là **subsequence** (bỏ số 1) nhưng **không** phải subarray (không liền nhau).
- `[7, 5]` **không** phải subsequence (đảo thứ tự).

Hàm `is_subsequence(sub, arr)` dưới đây dùng **hai con trỏ**: đi qua `arr`, mỗi khi gặp phần tử khớp với phần tử tiếp theo cần tìm trong `sub` thì tiến con trỏ của `sub`.

In [2]:
def is_subsequence(sub, arr):
    j = 0
    for x in arr:
        if j < len(sub) and x == sub[j]:
            j += 1
    return j == len(sub)

print(is_subsequence([5, 7], [5, 1, 7]))     # True
print(is_subsequence([7, 5], [5, 1, 7]))     # False
print(is_subsequence([3, 4, -1], [-1, -2, 3, 4]))  # False -> sai thứ tự

True
False
False


## 2. Nền tảng Python cần dùng

### 2.1. `sorted(..., key=..., reverse=True)`

`key` là hàm cho biết "sắp theo tiêu chí gì". `reverse=True` để giảm dần.

In [3]:
words = ["bb", "a", "ccc"]
print(sorted(words, key=len))                 # theo độ dài
print(sorted([3, 1, 2], reverse=True))        # giảm dần

['a', 'bb', 'ccc']
[3, 2, 1]


### 2.2. Sắp xếp **chỉ số** theo giá trị: `sorted(range(n), key=lambda i: nums[i])`

`lambda i: nums[i]` là hàm ẩn danh: nhận `i`, trả `nums[i]`. Kết quả là danh sách **chỉ số** đã sắp theo giá trị — ta vẫn nhớ vị trí gốc!

In [4]:
nums = [-1, -2, 3, 4]
idx = sorted(range(len(nums)), key=lambda i: nums[i], reverse=True)
print("chỉ số theo giá trị giảm dần:", idx)
print("giá trị tương ứng:", [nums[i] for i in idx])

chỉ số theo giá trị giảm dần: [3, 2, 0, 1]
giá trị tương ứng: [4, 3, -1, -2]


### 2.3. `enumerate` tạo cặp `(chỉ số, giá trị)` và sắp xếp tuple

Tuple được so sánh **lần lượt từng thành phần**. `sorted(enumerate(nums), key=lambda p: p[1])` sắp theo giá trị.

In [5]:
pairs = list(enumerate([3, 4, 3, 3]))
print(pairs)
print(sorted(pairs, key=lambda p: p[1], reverse=True))
print((1, 5) < (2, 0), (1, 5) < (1, 6))   # so sánh tuple

[(0, 3), (1, 4), (2, 3), (3, 3)]
[(1, 4), (0, 3), (2, 3), (3, 3)]
True True


### 2.4. Slicing `[:k]` và `sorted` lại theo chỉ số

In [6]:
top = [3, 2, 0]          # các chỉ số được chọn (chưa theo thứ tự)
print(sorted(top))       # [0, 2, 3] -> đúng thứ tự xuất hiện
print([10, 20, 30, 40][:2])

[0, 2, 3]
[10, 20]


### 2.5. `heapq` — hàng đợi ưu tiên (min-heap)

- `heapq.heappush(h, x)`: thêm `x`, `O(log size)`.
- `heapq.heappop(h)`: lấy ra phần tử **nhỏ nhất**, `O(log size)`.
- `h[0]`: xem phần tử nhỏ nhất.
- `heapq.nlargest(k, iterable, key=...)`: lấy `k` phần tử lớn nhất.

In [7]:
import heapq
h = []
for x in [5, 1, 4, 2]:
    heapq.heappush(h, x)
print("nhỏ nhất:", h[0])
print("pop:", heapq.heappop(h), "-> còn", sorted(h))
print("nlargest 2:", heapq.nlargest(2, [5, 1, 4, 2]))

nhỏ nhất: 1
pop: 1 -> còn [2, 4, 5]
nlargest 2: [5, 4]


## 3. Cách 1 — Trực giác: xóa dần phần tử nhỏ nhất

Gợi ý của đề: *"From a greedy perspective, what k elements should you pick?"* — dĩ nhiên là `k` phần tử **lớn nhất**.

Cách tự nhiên nhất để giữ thứ tự: bắt đầu từ cả mảng, **xóa phần tử nhỏ nhất** `n - k` lần. Phần còn lại tự giữ thứ tự gốc.

In [8]:
def max_subseq_remove_min(nums: list[int], k: int) -> list[int]:
    arr = nums[:]                 # copy, không sửa input
    for _ in range(len(nums) - k):
        arr.remove(min(arr))      # xóa lần xuất hiện ĐẦU TIÊN của giá trị nhỏ nhất
    return arr

for nums, k, ans in examples:
    print(max_subseq_remove_min(nums, k), "| tổng", sum(max_subseq_remove_min(nums, k)))

[3, 3] | tổng 6
[-1, 3, 4] | tổng 6
[4, 3] | tổng 7


**Giải thích cú pháp:** `nums[:]` tạo bản sao; `arr.remove(v)` xóa **lần xuất hiện đầu tiên** của `v` (mất `O(n)`); `min(arr)` cũng `O(n)`.

## 4. Nút thắt của baseline

Mỗi lần xóa tốn `O(n)` cho `min` và `O(n)` cho `remove` → tổng `O(n·(n - k))`, tệ nhất `O(n²)` (khi `k` nhỏ). Với `n = 1000` vẫn chạy, nhưng không đẹp.

Nhận xét: ta không cần xóa từng cái. Ta chỉ cần biết **tập chỉ số** của `k` phần tử lớn nhất — rồi đọc lại theo **thứ tự chỉ số tăng dần**.

Câu hỏi then chốt (gợi ý 2 của đề): *"Could you sort the array while maintaining the index?"*

## 5. Bước đột phá — Sắp xếp kèm chỉ số, chọn, rồi sắp lại theo chỉ số

Ba bước:

1. **Sắp chỉ số theo giá trị giảm dần** → biết phần tử nào lớn.
2. **Lấy `k` chỉ số đầu** → đó là vị trí của `k` phần tử lớn nhất.
3. **Sắp lại `k` chỉ số tăng dần** → khôi phục thứ tự gốc; đọc giá trị.

Ví dụ `nums = [-1, -2, 3, 4]`, `k = 3`:

```text
Bước 1: chỉ số theo giá trị giảm: [3, 2, 0, 1]   (4, 3, -1, -2)
Bước 2: lấy 3 cái đầu:            [3, 2, 0]
Bước 3: sắp lại tăng dần:         [0, 2, 3]  -> giá trị [-1, 3, 4]
```

In [9]:
nums, k = [-1, -2, 3, 4], 3
step1 = sorted(range(len(nums)), key=lambda i: nums[i], reverse=True)
step2 = step1[:k]
step3 = sorted(step2)
print(step1, step2, step3, [nums[i] for i in step3])

[3, 2, 0, 1] [3, 2, 0] [0, 2, 3] [-1, 3, 4]


## 6. Lời giải chính — mổ xẻ rồi ghép

### 6.1. Đoạn 1: chọn chỉ số của k phần tử lớn nhất

In [10]:
def top_k_indices(nums, k):
    order = sorted(range(len(nums)), key=lambda i: nums[i], reverse=True)
    return order[:k]

print(top_k_indices([3, 4, 3, 3], 2))

[1, 0]


### 6.2. Đoạn 2: khôi phục thứ tự gốc

In [11]:
def restore_order(nums, indices):
    return [nums[i] for i in sorted(indices)]

print(restore_order([3, 4, 3, 3], [1, 0]))

[3, 4]


### 6.3. Ghép lại

In [12]:
def max_subsequence(nums: list[int], k: int) -> list[int]:
    return restore_order(nums, top_k_indices(nums, k))

for nums, k, ans in examples:
    res = max_subsequence(nums, k)
    print(res, "tổng", sum(res), "| đáp án đề", ans)

[3, 3] tổng 6 | đáp án đề [3, 3]
[-1, 3, 4] tổng 6 | đáp án đề [-1, 3, 4]
[3, 4] tổng 7 | đáp án đề [3, 4]


### 6.4. Giá trị trùng có gây vấn đề không?

Với `[3, 4, 3, 3]`, `k = 2`: có ba số 3, ta chỉ cần **một**. Sắp xếp chỉ số chọn ra **đúng một** chỉ số của số 3 (vì mỗi chỉ số là một phần tử riêng). Python sắp xếp **ổn định** (stable): các phần tử bằng nhau giữ thứ tự gốc, nên chỉ số 0 được chọn. Đáp án `[3, 4]` hợp lệ.

## 7. TRACE từng bước

### 7.1. Bảng trace — `nums = [2, 1, 3, 3]`, `k = 2`

| Bước | Dữ liệu | Ý nghĩa |
|---|---|---|
| Cặp (i, v) | `(0,2) (1,1) (2,3) (3,3)` | gắn chỉ số |
| Sắp giảm theo v | `(2,3) (3,3) (0,2) (1,1)` | lớn trước |
| Lấy k=2 | `(2,3) (3,3)` | hai phần tử lớn nhất |
| Sắp theo i | `(2,3) (3,3)` | thứ tự gốc |
| Kết quả | `[3, 3]` | tổng 6 |

In [13]:
def trace_max_subsequence(nums, k):
    pairs = list(enumerate(nums))
    print("1. cặp (i, v):        ", pairs)
    by_val = sorted(pairs, key=lambda p: p[1], reverse=True)
    print("2. sắp giảm theo v:   ", by_val)
    chosen = by_val[:k]
    print(f"3. lấy k={k}:            ", chosen)
    by_idx = sorted(chosen)          # tuple so sánh theo i trước
    print("4. sắp lại theo i:    ", by_idx)
    res = [v for _, v in by_idx]
    print("5. kết quả:           ", res, "tổng =", sum(res))
    return res

trace_max_subsequence([2, 1, 3, 3], 2)
print()
trace_max_subsequence([-1, -2, 3, 4], 3)

1. cặp (i, v):         [(0, 2), (1, 1), (2, 3), (3, 3)]
2. sắp giảm theo v:    [(2, 3), (3, 3), (0, 2), (1, 1)]
3. lấy k=2:             [(2, 3), (3, 3)]
4. sắp lại theo i:     [(2, 3), (3, 3)]
5. kết quả:            [3, 3] tổng = 6

1. cặp (i, v):         [(0, -1), (1, -2), (2, 3), (3, 4)]
2. sắp giảm theo v:    [(3, 4), (2, 3), (0, -1), (1, -2)]
3. lấy k=3:             [(3, 4), (2, 3), (0, -1)]
4. sắp lại theo i:     [(0, -1), (2, 3), (3, 4)]
5. kết quả:            [-1, 3, 4] tổng = 6


[-1, 3, 4]

### 7.2. Trace cách heap kích thước k

Duyệt từng phần tử, đẩy `(giá trị, chỉ số)` vào min-heap; nếu heap vượt `k` thì **pop phần tử nhỏ nhất**. Cuối cùng heap chứa `k` phần tử lớn nhất.

In [14]:
def trace_heap(nums, k):
    h = []
    for i, v in enumerate(nums):
        heapq.heappush(h, (v, i))
        msg = f"push ({v},{i})"
        if len(h) > k:
            out = heapq.heappop(h)
            msg += f", heap > k nên pop {out}"
        print(f"  i={i}: {msg:35} heap={sorted(h)}")
    idx = sorted(i for _, i in h)
    print("  chỉ số giữ lại:", idx, "->", [nums[i] for i in idx])

trace_heap([-1, -2, 3, 4], 3)

  i=0: push (-1,0)                         heap=[(-1, 0)]
  i=1: push (-2,1)                         heap=[(-2, 1), (-1, 0)]
  i=2: push (3,2)                          heap=[(-2, 1), (-1, 0), (3, 2)]
  i=3: push (4,3), heap > k nên pop (-2, 1) heap=[(-1, 0), (3, 2), (4, 3)]
  chỉ số giữ lại: [0, 2, 3] -> [-1, 3, 4]


### 7.3. Trace cách "ngưỡng" (threshold) — xử lý trùng bằng đếm

Ý tưởng sáng tạo khác: tìm giá trị **nhỏ nhất được chọn** `t` (phần tử lớn thứ k). Đếm cần lấy bao nhiêu phần tử **bằng** `t` (`need_t`). Duyệt mảng gốc: lấy mọi phần tử `> t`, và lấy phần tử `== t` cho tới khi đủ `need_t`.

In [15]:
def trace_threshold(nums, k):
    top = sorted(nums, reverse=True)[:k]
    t = top[-1]
    need_t = top.count(t)
    print(f"  top k = {top}, ngưỡng t = {t}, cần lấy {need_t} phần tử bằng t")
    res = []
    for v in nums:
        if v > t:
            res.append(v); act = "> t: lấy"
        elif v == t and need_t > 0:
            res.append(v); need_t -= 1; act = f"== t: lấy (còn cần {need_t})"
        else:
            act = "bỏ"
        print(f"    v={v:>3}: {act:25} res={res}")
    return res

trace_threshold([3, 4, 3, 3], 2)

  top k = [4, 3], ngưỡng t = 3, cần lấy 1 phần tử bằng t
    v=  3: == t: lấy (còn cần 0)     res=[3]
    v=  4: > t: lấy                  res=[3, 4]
    v=  3: bỏ                        res=[3, 4]
    v=  3: bỏ                        res=[3, 4]


[3, 4]

## 8. Code nộp LeetCode (Python 3) — nhiều cách

### Cách A — Xóa dần phần tử nhỏ nhất (baseline)

In [16]:
class Solution:
    def maxSubsequence(self, nums: list[int], k: int) -> list[int]:
        arr = nums[:]
        for _ in range(len(nums) - k):
            arr.remove(min(arr))
        return arr

print([Solution().maxSubsequence(n, k) for n, k, _ in examples])

[[3, 3], [-1, 3, 4], [4, 3]]


### Cách B — Sắp xếp chỉ số (lời giải chính)

In [17]:
class Solution:
    def maxSubsequence(self, nums: list[int], k: int) -> list[int]:
        idx = sorted(range(len(nums)), key=lambda i: nums[i], reverse=True)[:k]
        return [nums[i] for i in sorted(idx)]

print([Solution().maxSubsequence(n, k) for n, k, _ in examples])

[[3, 3], [-1, 3, 4], [3, 4]]


### Cách C — `heapq.nlargest` trên cặp (chỉ số, giá trị)

In [18]:
import heapq

class Solution:
    def maxSubsequence(self, nums: list[int], k: int) -> list[int]:
        best = heapq.nlargest(k, enumerate(nums), key=lambda p: p[1])
        return [v for _, v in sorted(best)]

print([Solution().maxSubsequence(n, k) for n, k, _ in examples])

[[3, 3], [-1, 3, 4], [3, 4]]


### Cách D — Min-heap kích thước k (`O(n log k)`)

In [19]:
import heapq

class Solution:
    def maxSubsequence(self, nums: list[int], k: int) -> list[int]:
        h = []
        for i, v in enumerate(nums):
            heapq.heappush(h, (v, i))
            if len(h) > k:
                heapq.heappop(h)
        return [nums[i] for i in sorted(i for _, i in h)]

print([Solution().maxSubsequence(n, k) for n, k, _ in examples])

[[3, 3], [-1, 3, 4], [4, 3]]


### Cách E — Ngưỡng + đếm trùng (không cần lưu chỉ số)

In [20]:
class Solution:
    def maxSubsequence(self, nums: list[int], k: int) -> list[int]:
        top = sorted(nums, reverse=True)[:k]
        t = top[-1]
        need_t = top.count(t)
        res = []
        for v in nums:
            if v > t:
                res.append(v)
            elif v == t and need_t > 0:
                res.append(v)
                need_t -= 1
        return res

print([Solution().maxSubsequence(n, k) for n, k, _ in examples])

[[3, 3], [-1, 3, 4], [3, 4]]


## 9. Vì sao thuật toán đúng?

**Tham lam đúng (exchange argument):** giả sử một dãy con độ dài `k` chứa phần tử `a` **không** thuộc nhóm `k` lớn nhất, và bỏ qua phần tử `b` thuộc nhóm đó với `b >= a`. Thay `a` bằng `b` (rồi xếp lại theo thứ tự chỉ số) vẫn là một dãy con độ dài `k` và tổng **không giảm**. Lặp lại, ta đi tới dãy gồm đúng `k` phần tử lớn nhất — nên tổng của nó là **lớn nhất**.

**Thứ tự:** mọi tập chỉ số, khi đọc theo chỉ số tăng dần, đều là một subsequence hợp lệ. Vì vậy chỉ cần chọn **tập** đúng, rồi sắp chỉ số.

**Cách E:** các phần tử `> t` đều thuộc top k (lấy hết); phần tử `== t` cần đúng `need_t` cái; tổng số lấy = (số phần tử > t) + `need_t` = `k`. Tổng bằng `sum(top)`.

**Cách D (heap):** invariant — sau khi xử lý `i+1` phần tử đầu, heap chứa `min(k, i+1)` phần tử lớn nhất trong số đó (vì mỗi lần vượt k ta bỏ phần tử nhỏ nhất).

## 10. Phân tích Time / Space Complexity

| Cách | Time | Space phụ | Ghi chú |
|---|---|---|---|
| A. Xóa min | `O(n·(n−k))` | `O(n)` | dễ nghĩ |
| B. Sắp chỉ số | `O(n log n)` | `O(n)` | lời giải chính |
| C. `nlargest` | `O(n log k)` | `O(k)` | gọn |
| D. Min-heap k | `O(n log k)` | `O(k)` | tốt khi k nhỏ |
| E. Ngưỡng | `O(n log n)` (sort) | `O(n)` | có thể dùng quickselect để đạt `O(n)` trung bình |

Bước cuối "sắp lại k chỉ số" tốn `O(k log k)` ≤ `O(n log n)`.

## 11. Kiểm thử — bài có nhiều đáp án đúng

Không thể `assert res == [3, 4]` vì `[4, 3]` cũng đúng. Ta kiểm tra **tính chất**:
1. `len(res) == k`;
2. `res` là subsequence của `nums`;
3. `sum(res) == sum(sorted(nums)[-k:])` (tổng tối ưu).

In [21]:
import random

def valid_answer(nums, k, res):
    return (len(res) == k
            and is_subsequence(res, nums)
            and sum(res) == sum(sorted(nums)[-k:]))

def sol_A(n, k): return max_subseq_remove_min(n, k)
def sol_B(n, k): return max_subsequence(n, k)
def sol_C(n, k): return [v for _, v in sorted(heapq.nlargest(k, enumerate(n), key=lambda p: p[1]))]
def sol_D(n, k):
    h = []
    for i, v in enumerate(n):
        heapq.heappush(h, (v, i))
        if len(h) > k: heapq.heappop(h)
    return [n[i] for i in sorted(i for _, i in h)]
def sol_E(n, k):
    top = sorted(n, reverse=True)[:k]; t = top[-1]; need = top.count(t); res = []
    for v in n:
        if v > t: res.append(v)
        elif v == t and need: res.append(v); need -= 1
    return res

SOLS = [sol_A, sol_B, sol_C, sol_D, sol_E]
tests = [(n, k) for n, k, _ in examples] + [
    ([5], 1), ([-5, -1, -3], 1), ([-5, -1, -3], 3),
    ([1, 1, 1, 1], 2), ([7, 1, 7, 1, 7], 2), ([0, -1, 0], 2),
]
for n, k in tests:
    for f in SOLS:
        r = f(n, k)
        assert valid_answer(n, k, r), (f.__name__, n, k, r)
    print(f"PASS nums={n}, k={k} -> ví dụ đáp án {sol_B(n, k)}")

PASS nums=[2, 1, 3, 3], k=2 -> ví dụ đáp án [3, 3]
PASS nums=[-1, -2, 3, 4], k=3 -> ví dụ đáp án [-1, 3, 4]
PASS nums=[3, 4, 3, 3], k=2 -> ví dụ đáp án [3, 4]
PASS nums=[5], k=1 -> ví dụ đáp án [5]
PASS nums=[-5, -1, -3], k=1 -> ví dụ đáp án [-1]
PASS nums=[-5, -1, -3], k=3 -> ví dụ đáp án [-5, -1, -3]
PASS nums=[1, 1, 1, 1], k=2 -> ví dụ đáp án [1, 1]
PASS nums=[7, 1, 7, 1, 7], k=2 -> ví dụ đáp án [7, 7]
PASS nums=[0, -1, 0], k=2 -> ví dụ đáp án [0, 0]


In [22]:
rng = random.Random(2099)
for _ in range(500):
    n = [rng.randint(-5, 5) for _ in range(rng.randint(1, 12))]
    k = rng.randint(1, len(n))
    for f in SOLS:
        assert valid_answer(n, k, f(n, k)), (f.__name__, n, k)
print("500 test ngẫu nhiên: cả 5 cách đều cho đáp án hợp lệ")

500 test ngẫu nhiên: cả 5 cách đều cho đáp án hợp lệ


## 12. Lỗi người mới hay gặp — code sai, vì sao sai, cách sửa

### Lỗi 1: Sắp giá trị rồi trả luôn (mất thứ tự gốc)

In [23]:
def wrong_sorted_values(nums, k):
    return sorted(nums, reverse=True)[:k]       # SAI: thứ tự giảm dần

r = wrong_sorted_values([-1, -2, 3, 4], 3)
print("Sai :", r, "-> hợp lệ?", valid_answer([-1, -2, 3, 4], 3, r))
print("Đúng:", max_subsequence([-1, -2, 3, 4], 3))

Sai : [4, 3, -1] -> hợp lệ? False
Đúng: [-1, 3, 4]


### Lỗi 2: Lấy k giá trị lớn nhất rồi lọc mảng gốc bằng `in` (trùng giá trị → thừa phần tử)

In [24]:
def wrong_filter_in(nums, k):
    top = sorted(nums, reverse=True)[:k]
    return [v for v in nums if v in top]        # SAI khi có trùng

r = wrong_filter_in([3, 4, 3, 3], 2)
print("Sai :", r, "-> độ dài", len(r), "≠ k=2")

Sai : [3, 4, 3, 3] -> độ dài 4 ≠ k=2


**Sửa:** đếm số lần còn cần lấy (Cách E) hoặc làm việc với **chỉ số** (Cách B).

### Lỗi 3: Nhầm subsequence với subarray (tìm đoạn liên tiếp tổng lớn nhất)

In [25]:
def wrong_subarray(nums, k):
    best = max(range(len(nums) - k + 1), key=lambda s: sum(nums[s:s + k]))
    return nums[best:best + k]                  # SAI: chỉ xét đoạn liên tiếp

r = wrong_subarray([5, -100, 5], 2)
print("Sai :", r, "tổng", sum(r), "| Đúng:", max_subsequence([5, -100, 5], 2), "tổng", sum(max_subsequence([5, -100, 5], 2)))

Sai : [5, -100] tổng -95 | Đúng: [5, 5] tổng 10


### Lỗi 4: Sắp lại kết quả theo **giá trị** thay vì **chỉ số**

In [26]:
nums, k = [4, 1, 3], 2
idx = sorted(range(3), key=lambda i: nums[i], reverse=True)[:k]
print("Sai (sắp theo giá trị):", sorted(nums[i] for i in idx))
print("Đúng (sắp theo chỉ số):", [nums[i] for i in sorted(idx)])

Sai (sắp theo giá trị): [3, 4]
Đúng (sắp theo chỉ số): [4, 3]


## 13. Mẹo nhận dạng pattern

- "Chọn k phần tử để tổng/tích lớn nhất, không ràng buộc liền kề" → **tham lam: k phần tử lớn nhất**.
- "Giữ thứ tự gốc" → làm việc với **chỉ số**, rồi `sorted(indices)`.
- "k lớn nhất / nhỏ nhất" với n lớn, k nhỏ → **heap kích thước k** (`O(n log k)`).
- Giá trị trùng + chọn theo ngưỡng → **đếm số lần còn cần lấy** của ngưỡng.

Bài liên quan: 215 Kth Largest Element in an Array, 347 Top K Frequent Elements, 1985 Find the Kth Largest Integer in the Array.

## 14. Biến thể

### 14.1. Dãy con độ dài k có tổng **nhỏ nhất**

In [27]:
def min_subsequence_k(nums, k):
    idx = sorted(range(len(nums)), key=lambda i: nums[i])[:k]
    return [nums[i] for i in sorted(idx)]

print(min_subsequence_k([-1, -2, 3, 4], 2))

[-1, -2]


### 14.2. Chỉ cần **tổng lớn nhất** (không cần dãy)

In [28]:
def max_sum_k(nums, k):
    return sum(heapq.nlargest(k, nums))

print(max_sum_k([2, 1, 3, 3], 2), max_sum_k([-1, -2, 3, 4], 3))

6 6


### 14.3. Nếu yêu cầu dãy con **liên tiếp** (subarray) độ dài k tổng lớn nhất → cửa sổ trượt

In [29]:
def max_subarray_k(nums, k):
    window = sum(nums[:k]); best = window; start = 0
    for i in range(k, len(nums)):
        window += nums[i] - nums[i - k]      # trượt: thêm phải, bỏ trái
        if window > best:
            best, start = window, i - k + 1
    return nums[start:start + k]

print(max_subarray_k([5, -100, 5, 6], 2))

[5, 6]


## 15. Hệ thống luyện tập — 6 bài từ dễ đến nâng cao

Đáp án ở **mục 17**.

### Bài 1 — Chỉ số của phần tử lớn nhất (nếu trùng lấy chỉ số nhỏ nhất)

In [30]:
def argmax(nums):
    # TODO: viết code ở đây
    pass

# assert argmax([3, 7, 7, 1]) == 1

### Bài 2 — Kiểm tra subsequence (tự viết lại bằng hai con trỏ)

In [31]:
def is_subseq_ex(sub, arr):
    # TODO: viết code ở đây
    pass

# assert is_subseq_ex([1, 3], [1, 2, 3]) and not is_subseq_ex([3, 1], [1, 2, 3])

### Bài 3 — Tự viết lại lời giải #2099 bằng sắp xếp chỉ số

In [32]:
def max_subsequence_ex(nums, k):
    # TODO: viết code ở đây
    pass

# assert valid_answer([-1, -2, 3, 4], 3, max_subsequence_ex([-1, -2, 3, 4], 3))

### Bài 4 — Biến thể: dãy con độ dài k có **tích tuyệt đối** lớn nhất (theo `abs`)

In [33]:
def max_abs_subsequence(nums, k):
    # TODO: viết code ở đây
    pass

# assert max_abs_subsequence([-9, 1, 2, 8], 2) == [-9, 8]

### Bài 5 — Tự tạo test làm lộ **Lỗi 2** (`in` + trùng)

In [34]:
my_tests = [
    # TODO: (nums, k) mà wrong_filter_in trả độ dài khác k
]
# for n, k in my_tests:
#     assert len(wrong_filter_in(n, k)) != k

### Bài 6 — Mini challenge: k phần tử lớn nhất bằng min-heap tự viết bằng `heapq` (trả theo thứ tự gốc)
Yêu cầu: không dùng `sorted` trên toàn mảng, chỉ `heapq` với heap kích thước ≤ k.

In [35]:
def max_subsequence_heap_ex(nums, k):
    # TODO: viết code ở đây
    pass

# assert valid_answer([3, 4, 3, 3], 2, max_subsequence_heap_ex([3, 4, 3, 3], 2))

## 16. Gợi ý ba mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | duyệt giữ chỉ số tốt nhất | cập nhật khi `>` (không `>=`) | hoặc `max(range(n), key=nums.__getitem__)` |
| 2 | hai con trỏ | con trỏ `j` trên `sub` | tăng `j` khi khớp |
| 3 | chọn chỉ số | `sorted(range(n), key=..., reverse=True)[:k]` | `sorted(idx)` rồi đọc giá trị |
| 4 | đổi tiêu chí sắp | `key=lambda i: abs(nums[i])` | phần còn lại như bài 3 |
| 5 | lỗi khi trùng ở ngưỡng | nhiều phần tử bằng ngưỡng | `([1, 1, 1], 1)` |
| 6 | heap `(v, i)` | pop khi `len > k` | `sorted` chỉ k chỉ số ở cuối |

## 17. Đáp án tham khảo

In [36]:
def argmax(nums):
    best = 0
    for i in range(1, len(nums)):
        if nums[i] > nums[best]:
            best = i
    return best

def is_subseq_ex(sub, arr):
    j = 0
    for x in arr:
        if j < len(sub) and x == sub[j]:
            j += 1
    return j == len(sub)

def max_subsequence_ex(nums, k):
    idx = sorted(range(len(nums)), key=lambda i: nums[i], reverse=True)[:k]
    return [nums[i] for i in sorted(idx)]

def max_abs_subsequence(nums, k):
    idx = sorted(range(len(nums)), key=lambda i: abs(nums[i]), reverse=True)[:k]
    return [nums[i] for i in sorted(idx)]

my_tests = [([1, 1, 1], 1), ([3, 4, 3, 3], 2), ([5, 2, 2, 2], 2)]

def max_subsequence_heap_ex(nums, k):
    h = []
    for i, v in enumerate(nums):
        heapq.heappush(h, (v, i))
        if len(h) > k:
            heapq.heappop(h)
    return [nums[i] for i in sorted(i for _, i in h)]

assert argmax([3, 7, 7, 1]) == 1
assert is_subseq_ex([1, 3], [1, 2, 3]) and not is_subseq_ex([3, 1], [1, 2, 3])
assert valid_answer([-1, -2, 3, 4], 3, max_subsequence_ex([-1, -2, 3, 4], 3))
assert max_abs_subsequence([-9, 1, 2, 8], 2) == [-9, 8]
for n, k in my_tests:
    assert len(wrong_filter_in(n, k)) != k
assert valid_answer([3, 4, 3, 3], 2, max_subsequence_heap_ex([3, 4, 3, 3], 2))
print("Tất cả đáp án PASS")

Tất cả đáp án PASS


**Giải thích bằng lời:**
- Bài 1: cập nhật bằng `>` để giữ chỉ số nhỏ nhất khi trùng.
- Bài 2: mỗi phần tử của `sub` phải được "khớp" lần lượt theo thứ tự trong `arr`.
- Bài 3–4: chỉ thay tiêu chí `key`; phần "sắp lại theo chỉ số" giữ nguyên.
- Bài 5: Lỗi 2 sai khi ngưỡng (phần tử nhỏ nhất được chọn) xuất hiện nhiều hơn số cần lấy.
- Bài 6: heap luôn ≤ k phần tử nên mỗi thao tác `O(log k)`.

## 18. Tổng kết — template + checklist

**Template "chọn tập rồi giữ thứ tự":**
```python
idx = sorted(range(n), key=lambda i: tiêu_chí(nums[i]), reverse=True)[:k]
return [nums[i] for i in sorted(idx)]
```

**Checklist trước khi submit:**
- [ ] Kết quả giữ **thứ tự gốc** (sắp theo chỉ số, không theo giá trị).
- [ ] Đúng **k** phần tử, kể cả khi có giá trị trùng.
- [ ] Không nhầm subsequence với subarray.
- [ ] Xử lý số âm (vẫn chọn k lớn nhất, có thể toàn âm).
- [ ] Đúng tên method `maxSubsequence`.
- [ ] Test bằng **tính chất** vì đề chấp nhận nhiều đáp án.